In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

url = "https://www.bloomlatte.jp/categories/2646842"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
}

In [5]:
print("データ取得を開始します...")
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "lxml")

items = soup.find_all("a", class_="list")
print(f"取得した商品数: {len(items)}件\n")

data_list = []

for item in items:
    name_tag = item.find("p", class_="text1")
    name = name_tag.text.strip() if name_tag else "名前なし"
    
    link = item.get("href")
    all_text = item.get_text(separator=" ", strip=True)
    price_match = re.findall(r'¥\s*([\d,]+)', all_text)
    
    price = None
    if price_match:
        raw_price = price_match[-1].replace(",", "")
        try:
            price = int(raw_price)
        except ValueError:
            price = None

    data_list.append({
        "商品名": name,
        "価格": price,
        "URL": link
    })

データ取得を開始します...
取得した商品数: 30件



In [6]:
df = pd.DataFrame(data_list)
df = df.dropna(subset=['価格'])

print(df.head())

csv_filename = "base_items_tops.csv"
df.to_csv(csv_filename, index=False, encoding="utf-8-sig")

print(f"\n【完了】データを {csv_filename} に保存しました！")
print(f"平均価格: ¥{df['価格'].mean():.0f}")

                          商品名      価格  \
0  【30％OFF】◇ ゆったり スウェットプルパーカー  2912.0   
1       ◆  ドット柄 ボウタイブラウスリボンタイ  3170.0   
2     ◆  パール付きスカーフタイシャツブラウス上品  5100.0   
3     ◆  オフショルダーシャツタンクトップ重ね着風  3500.0   
4       ◆  ダブルジップニットカーディガン 羽織  5160.0   

                                         URL  
0   https://www.bloomlatte.jp/items/36495313  
1  https://www.bloomlatte.jp/items/137329116  
2  https://www.bloomlatte.jp/items/137329123  
3  https://www.bloomlatte.jp/items/137329130  
4  https://www.bloomlatte.jp/items/137329132  

【完了】データを base_items_tops.csv に保存しました！
平均価格: ¥4009
